# Hito 1 · Guía de supervivencia del veinteañero

¿Podemos encontrar el artículo del BOE que responde a una pregunta escrita con palabras cotidianas? En este primer hito lo probamos con dos buscadores, TF-IDF y BM25. Ambos buscan entre fragmentos. Para medir los aciertos miramos en qué puesto queda el artículo o disposición del que salió la pregunta.

## Entorno y configuración

Ejecuta desde la raíz del repositorio, con las dependencias de `requirements.txt`. El notebook lee los archivos de evaluación congelados y verifica que los fragmentos coinciden con el corpus. El script `descargar_corpus.py --comprobar` verifica también la reconstrucción desde los XML.

Los parámetros de fragmentación, del filtro por frecuencia documental y de BM25 se fijaron de antemano y no se ajustaron con las consultas de evaluación.

In [1]:
import json
import math
import statistics
from collections import Counter
from pathlib import Path

from descargar_corpus import dividir_documento, leer_jsonl

import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA = Path("data")
CORPUS = DATA / "corpus.jsonl"
CHUNKS = DATA / "chunks.jsonl"
MUESTRA = DATA / "muestra_consultas.jsonl"
RESPUESTAS = DATA / "respuestas_llm.jsonl"
QUERIES = DATA / "queries.jsonl"
RESPALDOS = DATA / "respaldo_consultas.json"

SEMILLA = 42
MIN_CONSULTAS = 30
K1 = 1.5
B = 0.75


## 1. Carga del corpus

Leemos el corpus generado a partir de las 11 copias del BOE. Antes de indexarlo comprobamos que cada documento tiene texto e identificador propio. El recuento por fuente y tema nos dice qué hemos incorporado realmente.

In [2]:
if not CORPUS.is_file():
    raise FileNotFoundError(f"No encuentro {CORPUS.resolve()}. Abre el notebook desde PLN_MIAAP.")

documentos = leer_jsonl(CORPUS)
if not documentos or any(not d["doc_id"] or not d["texto"].strip() for d in documentos):
    raise ValueError("El corpus está vacío o contiene documentos sin ID o sin texto.")

documentos_por_id = {d["doc_id"]: d for d in documentos}
if len(documentos_por_id) != len(documentos):
    raise ValueError("Hay doc_id duplicados en el corpus.")

print("Documentos:", len(documentos))
print("Por fuente:", dict(Counter(d["fuente"] for d in documentos)))
print("Por tema:", dict(Counter(d["tema"] for d in documentos)))

Documentos: 1395
Por fuente: {'BOE': 1395}
Por tema: {'trabajo': 164, 'renta': 223, 'nomina': 168, 'becas': 61, 'ocio': 23, 'consumo': 270, 'conducir': 486}


## 2. Unidad de indexación: fragmentación de los artículos

La **unidad de indexación** es el fragmento. Los artículos de hasta **300 palabras** quedan en una sola pieza. Los más largos se cortan en ventanas de **300 palabras**, con **45 palabras compartidas** entre ventanas consecutivas (**15 %**). El último fragmento puede ser más corto.

Cada pieza guarda su `chunk_id` propio, el `doc_id` del artículo y las posiciones de inicio y fin. Usamos solo su texto para buscar y conservamos los metadatos del corpus para mostrar los resultados. El archivo `data/chunks.jsonl` se puede reutilizar en el **Hito 2** sin cambiar la segmentación ni los IDs.

In [3]:
fragmentos = leer_jsonl(CHUNKS)
esperados = [f for d in documentos for f in dividir_documento(d)]
if fragmentos != esperados:
    raise ValueError("Los fragmentos guardados no corresponden al corpus y a la segmentación.")

fragmentos_por_id = {f["chunk_id"]: f for f in fragmentos}
if len(fragmentos_por_id) != len(fragmentos):
    raise ValueError("Hay chunk_id duplicados.")

longitudes = [f["fin_palabra"] - f["inicio_palabra"] for f in fragmentos]
print("Fragmentos:", len(fragmentos))
print(f"Longitud media: {statistics.mean(longitudes):.1f} palabras")
print("Longitud máxima:", max(longitudes), "palabras")

Fragmentos: 2315
Longitud media: 201.7 palabras
Longitud máxima: 300 palabras


## 3. Preprocesado con spaCy y selección del vocabulario

1. **Lematizar.** Convertimos cada palabra a su forma base en minúsculas. Por ejemplo, *compré* pasa a *comprar*.
2. **Filtrar por categoría gramatical.** Conservamos nombres, nombres propios, adjetivos, verbos y adverbios. También conservamos números, negaciones y unidades de plazo, aunque spaCy les asigne otra categoría. Quitamos los espacios, la puntuación y las *stopwords* salvo esas excepciones.
3. **Filtrar por frecuencia documental (DF).** Una vez procesados los fragmentos con spaCy, contamos en cuántos aparece cada lema. Por ejemplo, si un lema se repite diez veces en un mismo fragmento, cuenta una sola vez para DF. Conservamos los lemas que aparecen en **al menos 2 fragmentos** y en **como máximo el 95 %** de los fragmentos.

El vocabulario se aprende **solo con los fragmentos** y se aplica después tanto al corpus como a todas las preguntas. Si un número, una negación o un plazo se conserva con spaCy, aún puede quedar fuera por frecuencia documental. No se eliminan fragmentos vacíos tras el filtro. Así se conservan los IDs y el tamaño de la colección.

In [4]:
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
print(f"spaCy {spacy.__version__} | es_core_news_sm {nlp.meta['version']}")
CATEGORIAS = {"NOUN", "PROPN", "ADJ", "VERB", "ADV"}
NEGACIONES = {"no", "ni", "nunca", "jamás", "sin", "tampoco"}
UNIDADES_PLAZO = {"día", "semana", "mes", "año", "hora"}


def extraer_tokens(doc_spacy):
    tokens = []
    for token in doc_spacy:
        if token.is_space or token.is_punct:
            continue

        lema = (token.lemma_ or token.text).lower()
        es_excepcion = (
            token.lower_ in NEGACIONES
            or lema in UNIDADES_PLAZO
            or token.like_num
            or token.pos_ == "NUM"
        )

        if es_excepcion or (token.pos_ in CATEGORIAS and not token.is_stop):
            tokens.append(lema)

    return tokens


pregunta_prueba = "¿Puedo devolver los productos que compré hace 14 días sin recibo?"
print("Tras spaCy, antes de filtrar por DF:", extraer_tokens(nlp(pregunta_prueba)))

spaCy 3.8.11 | es_core_news_sm 3.8.0
Tras spaCy, antes de filtrar por DF: ['devolver', 'producto', 'comprar', '14', 'día', 'sin', 'recibo']


In [5]:
MIN_DF = 2
MAX_DF = 0.95

# Procesamos primero los textos con spaCy.
textos = (fragmento["texto"] for fragmento in fragmentos)
tokens_sin_filtrar_df = []
for doc_spacy in nlp.pipe(textos, batch_size=32):
    tokens_sin_filtrar_df.append(extraer_tokens(doc_spacy))

# Contamos cada lema como máximo una vez por fragmento.
df = Counter()
for tokens in tokens_sin_filtrar_df:
    df.update(set(tokens))

n_fragmentos = len(fragmentos)
vocabulario = set()
for lema, frecuencia in df.items():
    if frecuencia >= MIN_DF and frecuencia / n_fragmentos <= MAX_DF:
        vocabulario.add(lema)

if not vocabulario:
    raise ValueError("El filtro por DF dejó vacío el vocabulario. Revisad los umbrales.")


def filtrar_por_df(tokens):
    return [token for token in tokens if token in vocabulario]


def preprocesar(texto):
    return filtrar_por_df(extraer_tokens(nlp(texto)))


tokens_fragmentos = [filtrar_por_df(tokens) for tokens in tokens_sin_filtrar_df]
print("Fragmentos procesados:", len(tokens_fragmentos))
print(f"Términos antes del filtro DF: {len(df)} | después: {len(vocabulario)}")
print("Fragmentos sin términos tras filtrar:", sum(not tokens for tokens in tokens_fragmentos))
print("Pregunta tras todo el preprocesado:", preprocesar(pregunta_prueba))

Fragmentos procesados: 2315
Términos antes del filtro DF: 8825 | después: 5926
Fragmentos sin términos tras filtrar: 0
Pregunta tras todo el preprocesado: ['devolver', 'producto', 'comprar', '14', 'día', 'sin', 'recibo']


## 4. Recuperación de información

### 4.1. Cómo elegimos las preguntas

Sacamos al azar 50 fragmentos de la colección original con semilla 42. Así podíamos descartar casos ambiguos y conservar al menos 30. Guardamos esa muestra para saber de dónde salió cada pregunta. Tras limpiar el BOE conservamos los 30 casos y comprobamos que el fragmento de referencia de cada uno seguía respaldando la pregunta.

In [6]:
muestra = leer_jsonl(MUESTRA)
muestra_por_id = {f["chunk_id"]: f for f in muestra}
if len(muestra) != 50 or len(muestra_por_id) != len(muestra):
    raise ValueError("Se esperaban los 50 candidatos históricos con IDs únicos.")

print("Candidatos en la muestra histórica:", len(muestra))
print("Semilla del muestreo histórico:", SEMILLA)
print("Candidatos que conservan exactamente el mismo fragmento:",
      sum(f == fragmentos_por_id.get(f["chunk_id"]) for f in muestra))

Candidatos en la muestra histórica: 50
Semilla del muestreo histórico: 42
Candidatos que conservan exactamente el mismo fragmento: 29


#### Generación y revisión de las preguntas

Usamos ChatGPT para preparar los borradores. Le dimos esta instrucción:

> Para cada fragmento, escribe una pregunta natural en español que haría una persona para encontrar esa información. Basa la pregunta solo en el fragmento. **No copies las palabras ni las expresiones distintivas del texto**: reformula con tus propias palabras. No menciones el número del artículo ni el identificador en la pregunta. Devuelve una línea JSON por fragmento, con `chunk_id` y `query`. Conserva el identificador exactamente.

Tras una primera evaluación revisamos a mano las 30 preguntas. Buscamos que cada una describiera un caso concreto con palabras propias y que su fragmento de origen permitiera responderla. Reformulamos 11. En el apartado 4.5 se comparan las métricas con ambas versiones. `data/respuestas_llm.jsonl` conserva la redacción del LLM y `data/queries.jsonl` la revisada. En `REVISION_CONSULTAS.md` están los criterios y las decisiones.

La celda siguiente comprueba que cada consulta procede de un fragmento de la muestra con respuesta del LLM y que su documento relevante es el de ese fragmento. Si la limpieza cambió el fragmento, exige que el pasaje guardado en `data/respaldo_consultas.json` aparezca en las dos versiones.

`relevant_ids` y `source_chunk` son etiquetas que se utilizan después de ordenar los resultados.

In [7]:
def validar_consultas(consultas):
    if len(consultas) < MIN_CONSULTAS:
        raise ValueError(f"Se necesitan al menos {MIN_CONSULTAS} consultas.")
    qids = [q["qid"] for q in consultas]
    if len(set(qids)) != len(qids):
        raise ValueError("Hay qid duplicados.")

    respuestas = {r["chunk_id"]: r["query"] for r in leer_jsonl(RESPUESTAS)}
    respaldos = {r["qid"]: r for r in json.loads(RESPALDOS.read_text(encoding="utf-8"))}
    if not respaldos.keys() <= set(qids):
        raise ValueError("Hay pasajes de respaldo de consultas que no existen.")

    origen_en_muestra = {}
    for q in consultas:
        qid, cid = q["qid"], q["source_chunk"]
        respaldo = respaldos.get(qid)
        cid_muestra = respaldo["fragmento_muestra"] if respaldo else cid
        anterior = muestra_por_id.get(cid_muestra)
        actual = fragmentos_por_id.get(cid)
        if anterior is None or actual is None or cid_muestra not in respuestas:
            raise ValueError(f"{qid}: el fragmento no está en la muestra, en el índice o en las respuestas del LLM.")
        if respaldo and respaldo["fragmento_actual"] != cid:
            raise ValueError(f"{qid}: el pasaje de respaldo apunta a otro fragmento.")
        if q["relevant_ids"] != [actual["doc_id"]] or actual["doc_id"] != anterior["doc_id"]:
            raise ValueError(f"{qid}: el documento relevante difiere del documento de origen.")
        if anterior != actual:
            pasaje = respaldo["pasaje"] if respaldo else None
            if not pasaje or pasaje not in anterior["texto"] or pasaje not in actual["texto"]:
                raise ValueError(f"{qid}: el fragmento cambió y falta un pasaje que lo respalde en ambas versiones.")
        origen_en_muestra[qid] = cid_muestra

    if set(origen_en_muestra.values()) != respuestas.keys():
        raise ValueError("Las consultas no corresponden a las respuestas del LLM.")
    return [q["qid"] for q in consultas if q["query"] != respuestas[origen_en_muestra[q["qid"]]]]


consultas = leer_jsonl(QUERIES)
reformuladas = validar_consultas(consultas)
print("Consultas verificadas contra la muestra, las respuestas del LLM y los pasajes de respaldo:", len(consultas))
print(f"Reformuladas respecto al LLM ({len(reformuladas)}):", ", ".join(reformuladas))

Consultas verificadas contra la muestra, las respuestas del LLM y los pasajes de respaldo: 30
Reformuladas respecto al LLM (11): q001, q004, q009, q014, q015, q021, q022, q023, q027, q028, q030


### 4.2. Recuperación con TF-IDF y similitud coseno

Ambos reciben **los mismos tokens de los mismos fragmentos**. Todos los cálculos de IDF se hacen a nivel de fragmento, no a nivel de artículo completo. La media de longitud para BM25 también se calcula sobre fragmentos, en tokens tras el filtro.

Usamos `TfidfVectorizer` de scikit-learn con frecuencia logarítmica (`sublinear_tf=True`) e IDF suavizada. Le pasamos los tokens que ya produjo spaCy, sin que scikit-learn vuelva a tokenizar. Ajustamos el vectorizador **solo con los fragmentos** y usamos ese mismo modelo para transformar preguntas.

In [8]:
def usar_tokens(tokens):
    return tokens


tfidf = TfidfVectorizer(analyzer=usar_tokens, sublinear_tf=True)
matriz_tfidf = tfidf.fit_transform(tokens_fragmentos)


def puntuar_tfidf(tokens_pregunta):
    vector_pregunta = tfidf.transform([tokens_pregunta])
    return cosine_similarity(vector_pregunta, matriz_tfidf)[0]


print("Matriz TF-IDF (fragmentos × términos):", matriz_tfidf.shape)

Matriz TF-IDF (fragmentos × términos): (2315, 5926)


### 4.3. Recuperación con BM25

Implementamos **BM25 Okapi en Python** con `k1 = 1.5`, `b = 0.75` e IDF `log(1 + (N - df + 0.5) / (df + 0.5))`. Aquí `N` es el número de fragmentos y `df` indica en cuántos aparece el término.

Para puntuar cada fragmento recorremos los términos únicos de la pregunta. BM25 reduce el premio por repetir muchas veces un término y ajusta la puntuación según la longitud del fragmento.

In [9]:
frecuencias = [Counter(tokens) for tokens in tokens_fragmentos]
longitudes_tokens = [len(tokens) for tokens in tokens_fragmentos]
longitud_media = statistics.mean(longitudes_tokens)

frecuencia_documental = Counter()
for conteos in frecuencias:
    frecuencia_documental.update(conteos.keys())  # Una vez por término y fragmento.

n_fragmentos = len(fragmentos)
idf_bm25 = {}
for termino, df_termino in frecuencia_documental.items():
    idf_bm25[termino] = math.log1p((n_fragmentos - df_termino + 0.5) / (df_termino + 0.5))


def puntuar_bm25(tokens_pregunta):
    terminos = list(dict.fromkeys(tokens_pregunta))
    puntuaciones = []

    for conteos, longitud in zip(frecuencias, longitudes_tokens):
        ajuste = K1 * (1 - B + B * longitud / longitud_media)
        puntuacion = 0.0

        for termino in terminos:
            frecuencia = conteos[termino]
            if frecuencia > 0:
                puntuacion += idf_bm25[termino] * frecuencia * (K1 + 1) / (frecuencia + ajuste)

        puntuaciones.append(puntuacion)

    return puntuaciones


print(f"BM25: k1={K1}, b={B} | media: {longitud_media:.1f} tokens por fragmento")

BM25: k1=1.5, b=0.75 | media: 95.9 tokens por fragmento


### 4.4. Cómo pasamos de fragmentos a documentos

Ordenamos los fragmentos con puntuación positiva de mayor a menor. En caso de empate usamos el `chunk_id`. La primera vez que aparece un documento ya tenemos su mejor fragmento, así que podemos saltarnos las apariciones siguientes. Sobre esa lista de documentos únicos aplicamos el corte K.

In [10]:
def ordenar_fragmentos(puntuaciones):
    candidatos = []
    for fragmento, puntuacion in zip(fragmentos, puntuaciones):
        if puntuacion > 0:
            candidatos.append((fragmento, float(puntuacion)))

    return sorted(candidatos, key=lambda par: (-par[1], par[0]["chunk_id"]))


def agrupar_por_documento(ranking_fragmentos):
    vistos = set()
    ranking_documentos = []

    for fragmento, puntuacion in ranking_fragmentos:
        doc_id = fragmento["doc_id"]
        if doc_id not in vistos:
            vistos.add(doc_id)
            ranking_documentos.append((doc_id, puntuacion))

    return ranking_documentos


def recuperar_tfidf(tokens_pregunta):
    return ordenar_fragmentos(puntuar_tfidf(tokens_pregunta))


def recuperar_bm25(tokens_pregunta):
    return ordenar_fragmentos(puntuar_bm25(tokens_pregunta))


METODOS = {"TF-IDF": recuperar_tfidf, "BM25": recuperar_bm25}


def buscar(tokens_pregunta, metodo):
    ranking_fragmentos = METODOS[metodo](tokens_pregunta)
    ranking_documentos = agrupar_por_documento(ranking_fragmentos)
    return ranking_fragmentos, ranking_documentos


pregunta_demo = "¿Puedo devolver un producto comprado por internet?"
tokens_demo = preprocesar(pregunta_demo)

print("Consulta de demostración (fuera de la evaluación):", pregunta_demo)
for metodo in METODOS:
    _, resultados = buscar(tokens_demo, metodo)
    print("\n", metodo)
    for puesto, (doc_id, puntuacion) in enumerate(resultados[:3], start=1):
        articulo = documentos_por_id[doc_id]
        print(f"{puesto}. {doc_id} | {articulo.get('seccion', '')} | {puntuacion:.4f}")

Consulta de demostración (fuera de la evaluación): ¿Puedo devolver un producto comprado por internet?

 TF-IDF
1. BOE-A-2007-20555_a6 | Artículo 6 | 0.2205
2. BOE-A-2002-13758_dasegunda | Disposición adicional segunda | 0.1827
3. BOE-A-2007-20555_a142 | Artículo 142 | 0.1740

 BM25
1. BOE-A-2007-20555_a93 | Artículo 93 | 8.9371
2. BOE-A-2026-10885_a7 | Artículo 7 | 8.6365
3. BOE-A-2007-20555_a75 | Artículo 75 | 7.4966


### 4.5. Evaluación por documento

Para cada consulta dividimos el número de documentos relevantes recuperados en los primeros K puestos por el número de relevantes conocidos. Aquí hay una etiqueta por consulta, así que Recall@K es la proporción de consultas con acierto en K.

MRR promedia el recíproco del puesto del primer relevante. Usa todo el ranking de puntuación positiva, sin corte K. Si el relevante queda fuera de ese ranking, aporta cero. Las consultas sin términos conocidos devuelven una lista vacía.

Como medida adicional, Fragmento@5 y Fragmento@10 comprueban si aparece el fragmento que respalda cada pregunta en el ranking de fragmentos, antes de agrupar.

In [11]:
def recall_at_k(ranking, relevantes, k):
    recuperados = set(ranking[:k])
    return len(relevantes.intersection(recuperados)) / len(relevantes)


def primer_puesto_relevante(ranking, relevantes):
    for puesto, doc_id in enumerate(ranking, start=1):
        if doc_id in relevantes:
            return puesto
    return None


def rango_reciproco(puesto):
    return 1 / puesto if puesto is not None else 0.0


def mrr(puestos):
    return statistics.mean(rango_reciproco(puesto) for puesto in puestos)


def evaluar_consulta(consulta, tokens, metodo):
    ranking_fragmentos, ranking_documentos = buscar(tokens, metodo)
    ids_docs = [doc_id for doc_id, _ in ranking_documentos]
    ids_chunks = [fragmento["chunk_id"] for fragmento, _ in ranking_fragmentos]
    relevantes = set(consulta["relevant_ids"])

    puesto = primer_puesto_relevante(ids_docs, relevantes)

    fila = {
        "consulta": consulta,
        "puesto": puesto,
        "top5": ids_docs[:5],
        "RR": rango_reciproco(puesto),
    }
    for k in (5, 10):
        fila[f"Recall@{k}"] = recall_at_k(ids_docs, relevantes, k)
        fila[f"Fragmento@{k}"] = float(consulta["source_chunk"] in ids_chunks[:k])

    return fila

In [12]:
# Los empates siguen el chunk_id. Los ceros quedan fuera y la deduplicación va antes de K.
puntuaciones_prueba = [0.0] * len(fragmentos)
indices = next([i, i + 1, i + 2] for i in range(len(fragmentos) - 2)
               if fragmentos[i]["doc_id"] == fragmentos[i + 1]["doc_id"]
               and fragmentos[i]["doc_id"] != fragmentos[i + 2]["doc_id"])
for indice, puntuacion in zip(indices, (2.0, 2.0, 1.0)):
    puntuaciones_prueba[indice] = puntuacion
ranking_prueba = ordenar_fragmentos(puntuaciones_prueba)
assert [f["chunk_id"] for f, _ in ranking_prueba] == [fragmentos[i]["chunk_id"] for i in indices]
assert agrupar_por_documento(ranking_prueba)[:2] == [
    (fragmentos[indices[0]]["doc_id"], 2.0), (fragmentos[indices[2]]["doc_id"], 1.0)]
assert all(buscar([], metodo) == ([], []) for metodo in METODOS)
assert recall_at_k(["a", "b"], {"b"}, 1) == 0
assert recall_at_k(["a", "b"], {"b"}, 2) == 1
assert mrr([2, None]) == 0.25
print("Comprobaciones de ranking y métricas superadas.")

resultados_por_metodo = {}
tokens_consultas = [preprocesar(q["query"]) for q in consultas]
print("Consultas evaluadas:", len(consultas))
print(f"{'Método':<10} {'Recall@5':>10} {'Recall@10':>10} {'MRR':>8} {'Frag.@5':>9} {'Frag.@10':>9}")
for metodo in METODOS:
    filas = [evaluar_consulta(q, tokens, metodo) for q, tokens in zip(consultas, tokens_consultas)]
    resultados_por_metodo[metodo] = filas
    valores = [statistics.mean(fila[m] for fila in filas)
               for m in ("Recall@5", "Recall@10", "RR", "Fragmento@5", "Fragmento@10")]
    print(f"{metodo:<10} " + " ".join(f"{valor:>9.4f}" for valor in valores))

Comprobaciones de ranking y métricas superadas.


Consultas evaluadas: 30
Método       Recall@5  Recall@10      MRR   Frag.@5  Frag.@10


TF-IDF        0.5667    0.6667    0.3459    0.4333    0.5000


BM25          0.4667    0.6667    0.3608    0.4000    0.4667


#### Comparación con las preguntas originales del LLM

Las preguntas se revisaron después de una primera evaluación. Para que se vea el efecto de esa revisión, repetimos la evaluación con la redacción original de `data/respuestas_llm.jsonl`. Los fragmentos y documentos relevantes son los mismos.

In [13]:
respuestas_originales = {r["chunk_id"]: r["query"] for r in leer_jsonl(RESPUESTAS)}
fragmento_en_muestra = {r["qid"]: r["fragmento_muestra"]
                        for r in json.loads(RESPALDOS.read_text(encoding="utf-8"))}
consultas_originales = [
    dict(q, query=respuestas_originales[fragmento_en_muestra.get(q["qid"], q["source_chunk"])])
    for q in consultas
]

print(f"{'Método':<10} {'Versión':<10} {'Recall@5':>9} {'Recall@10':>10} {'MRR':>8}")
for metodo in METODOS:
    for version, lista in (("original", consultas_originales), ("revisada", consultas)):
        filas = [evaluar_consulta(q, preprocesar(q["query"]), metodo) for q in lista]
        valores = [statistics.mean(fila[m] for fila in filas) for m in ("Recall@5", "Recall@10", "RR")]
        print(f"{metodo:<10} {version:<10} " + " ".join(f"{valor:>9.4f}" for valor in valores))

Método     Versión     Recall@5  Recall@10      MRR


TF-IDF     original      0.5667    0.6333    0.4174
TF-IDF     revisada      0.5667    0.6667    0.3459


BM25       original      0.6667    0.7333    0.4687


BM25       revisada      0.4667    0.6667    0.3608


### 4.6. Fallos compartidos

Contamos un fallo compartido cuando los dos buscadores dejan el documento esperado fuera de sus diez primeros resultados. La tabla muestra su puesto y qué términos sobreviven al preprocesado. El solapamiento compara la consulta con su fragmento de referencia. El puesto del documento puede depender de otro fragmento.

In [14]:
def fallos_compartidos(k):
    pares = zip(resultados_por_metodo["TF-IDF"], resultados_por_metodo["BM25"])
    return [(tf, bm) for tf, bm in pares if tf[f"Recall@{k}"] == bm[f"Recall@{k}"] == 0]


fallos_top10 = fallos_compartidos(10)
fallos_top5 = fallos_compartidos(5)
print("Fallos de ambos en Top 10:", len(fallos_top10))
print("Fallos de ambos en Top 5:", len(fallos_top5))
for fila_tf, fila_bm in fallos_top10 or fallos_top5:
    q = fila_tf["consulta"]
    origen = fragmentos_por_id[q["source_chunk"]]
    tokens_base = extraer_tokens(nlp(q["query"]))
    consulta = set(filtrar_por_df(tokens_base))
    comunes = consulta.intersection(preprocesar(origen["texto"]))
    descartados_df = {t for t in tokens_base if t in df and t not in vocabulario}
    ausentes = {t for t in tokens_base if t not in df}
    print(f"\n{q['qid']} · {q['query']}")
    print(f"  Origen: {origen['chunk_id']} | TF-IDF: {fila_tf['puesto']} | BM25: {fila_bm['puesto']}")
    print("  Lemas en común:", ", ".join(sorted(comunes)) or "ninguno")
    print("  Lemas excluidos por DF:", ", ".join(sorted(descartados_df)) or "ninguno")
    print("  Lemas ausentes tras spaCy en la colección:", ", ".join(sorted(ausentes)) or "ninguno")
    for metodo, fila in (("TF-IDF", fila_tf), ("BM25", fila_bm)):
        print(f"  Primeros resultados de {metodo}:")
        for doc_id in fila["top5"][:2]:
            print(f"    {doc_id} | {documentos_por_id[doc_id]['seccion']}")

Fallos de ambos en Top 10: 9
Fallos de ambos en Top 5: 13

q002 · Soy autónoma y voy a declarar de una vez lo que gané con un encargo puntual que me llevó más de dos años. ¿En qué porcentaje puedo rebajar ese importe en la renta?
  Origen: BOE-A-2006-20764_a32_c0001 | TF-IDF: 220 | BM25: 154
  Lemas en común: año, dos, importe, renta
  Lemas excluidos por DF: ganar
  Lemas ausentes tras spaCy en la colección: ninguno
  Primeros resultados de TF-IDF:
    BOE-A-2015-11430_a4 | Artículo 4
    BOE-A-2008-821_a40 | Artículo 40
  Primeros resultados de BM25:
    BOE-A-2008-821_a40 | Artículo 40
    BOE-A-2006-20764_a18 | Artículo 18

q003 · Si las autoridades cortan las carreteras por una inundación y no puedo llegar al trabajo, ¿cuántos días puedo faltar al principio con ese permiso?
  Origen: BOE-A-2015-11430_a37_c0003 | TF-IDF: 43 | BM25: 14
  Lemas en común: autoridad, día, permiso, trabajo
  Lemas excluidos por DF: faltar
  Lemas ausentes tras spaCy en la colección: cortir
  Primeros re

### 4.7. Comparación de resultados y limitaciones

TF-IDF encuentra 17 documentos relevantes entre los cinco primeros y BM25 encuentra 14. A diez puestos, ambos aciertan 20 de 30, aunque las consultas acertadas difieren. BM25 tiene un MRR ligeramente superior (0,3608 frente a 0,3459). Con 30 consultas, cada acierto cambia Recall en 0,0333. Con las preguntas originales del LLM la comparación era otra: BM25 superaba a TF-IDF en las tres métricas (apartado 4.5). La revisión, que quitó términos copiados de la ley, perjudica más a BM25.

- **Diferencias de vocabulario.** En q012 solo coincide `día` con el fragmento de origen. La pregunta usa «muere» y la ley «fallecimiento». En q024 el fragmento habla de «misiones diplomáticas» y «embajada» queda fuera del vocabulario. BM25 coloca primero el artículo 9 del IRPF, que trata de un supuesto de residencia distinto al del artículo 10.
- **Lematización y vocabulario.** En q003 spaCy devuelve `cortir` a partir de «cortan». Entre los primeros resultados aparecen permisos de conducción. En q009 el corte por frecuencia elimina, entre otros, `desgravación` y `rebaja`. Estas observaciones identifican pérdidas de información. El efecto de cambiar cada filtro requeriría un experimento separado.
- **Coincidencias con otro significado.** En q030 ambos métodos devuelven primero artículos sobre anualidades por alimentos. La pregunta trata de etiquetado y alérgenos. El lema `alimento` sobrevive en la consulta, pero la única coincidencia con su fragmento es `no`. El filtro elimina `etiqueta`.
- **Resultados relacionados.** En q019 ambos métodos devuelven primero los artículos 137 y 138 de la misma norma, que definen el producto defectuoso y el productor. La cuantía que pide la consulta figura en el artículo 141. En q002, el artículo 18 del IRPF recuperado por BM25 habla de rendimientos del trabajo, mientras la consulta se refiere a una actividad económica.
- **Otros fallos compartidos.** En q013 se conservan «semáforo» y «quedar», pero faltan `atasco` y `bloquear` del vocabulario. En q025 quedan fuera `casa` y `venderte`, y solo coinciden `administración` y `sin` con el fragmento que respalda la consulta.

La evaluación mide si se recupera el documento de origen, con una etiqueta por consulta. La recuperación densa se puede contrastar sobre estos mismos casos para comprobar si mejora las búsquedas con paráfrasis.

## Anexo. Comprobación de coincidencias literales

La revisión manual y la comprobación de los fragmentos se recogen en `REVISION_CONSULTAS.md`. Esta comprobación señala trigramas con al menos dos palabras de contenido que aparecen literalmente en la pregunta y en el fragmento. Un aviso requiere interpretación, pues algunas fechas o expresiones comunes pueden coincidir.

In [15]:
PALABRAS_VACIAS = {"de", "la", "el", "en", "y", "a", "que", "los", "las", "del", "se", "por",
                   "un", "una", "con", "para", "al", "lo", "su", "no", "es", "o", "mi", "me",
                   "si", "qué", "cómo", "puedo", "tengo", "ya"}


def palabras(texto):
    return [t.lower_ for t in nlp.make_doc(texto) if t.is_alpha or t.like_num]


def trigramas_copiados(consulta):
    q = palabras(consulta["query"])
    texto = " " + " ".join(palabras(fragmentos_por_id[consulta["source_chunk"]]["texto"])) + " "
    copias = []
    for i in range(len(q) - 2):
        trigrama = q[i:i + 3]
        contenido = sum(p not in PALABRAS_VACIAS for p in trigrama)
        if contenido >= 2 and f" {' '.join(trigrama)} " in texto:
            copias.append(" ".join(trigrama))
    return copias


consultas_revision = leer_jsonl(QUERIES)
con_copias = {c["qid"]: trigramas_copiados(c) for c in consultas_revision}
con_copias = {qid: copias for qid, copias in con_copias.items() if copias}
print("Consultas comprobadas automáticamente:", len(consultas_revision))
print("Consultas con algún trigrama literal del fragmento:", len(con_copias))
for qid, copias in con_copias.items():
    print(f"  {qid}: {copias}")

Consultas comprobadas automáticamente: 30
Consultas con algún trigrama literal del fragmento: 1
  q005: ['agosto de 2012']


Contrastamos los avisos de esta comprobación con el texto. En q005, «agosto de 2012» identifica el periodo por el que se pregunta.